In [46]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_predict
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [47]:
print("Loading MNIST...")

mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False
)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

print("Completed")
print(X.shape)

Loading MNIST...
Completed
(70000, 784)


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(56000, 784)
(14000, 784)


In [49]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [50]:
base_models = {

    "lr": (
        LogisticRegression(),
        {
            "C":[0.1,1],
            "max_iter":[1000]
        }
    ),

    "knn": (
        KNeighborsClassifier(),
        {
            "n_neighbors":[3,5]
        }
    ),

    "svm": (
    LinearSVC(
        max_iter=5000,
        dual=False
    ),
    {
        "C": [0.1]
    }
),

    "dt": (
        DecisionTreeClassifier(),
        {
            "max_depth":[10]
        }
    ),

    "rf": (
        RandomForestClassifier(),
        {
            "n_estimators":[100],
            "max_depth":[20]
        }
    )
}

In [ ]:
n_models = len(base_models)

meta_train = np.zeros(
    (X_train_scaled.shape[0], n_models)
)

meta_test = np.zeros(
    (X_test_scaled.shape[0], n_models)
)

best_models = {}
model_scores = {}

outer_cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

In [53]:
for idx, (name, (model, params)) in enumerate(base_models.items()):

    print("\n" + "=" * 60)
    print(f"Training {name.upper()}")
    print("=" * 60)

    # Grid Search
    grid = GridSearchCV(
        estimator=model,
        param_grid=params,
        cv=3,
        scoring="accuracy",
        n_jobs=3
    )

    grid.fit(
        X_train_scaled,
        y_train
    )

    best_model = grid.best_estimator_

    best_models[name] = best_model

    print("Best Parameters:")
    print(grid.best_params_)

    # OOF probabilities
    oof_probs = cross_val_predict(
        best_model,
        X_train_scaled,
        y_train,
        cv=outer_cv,
        method="predict_proba",
        n_jobs=3
    )

    oof_max_prob = np.max(
        oof_probs,
        axis=1
    )

    meta_train[:, idx] = oof_max_prob

    # Train on full training data
    best_model.fit(
        X_train_scaled,
        y_train
    )

    # Test predictions
    y_test_pred = best_model.predict(
        X_test_scaled
    )

    acc = accuracy_score(
        y_test,
        y_test_pred
    )

    model_scores[name] = acc

    print(f"{name.upper()} Accuracy: {acc:.4f}")

    # Test probabilities for stacking
    test_probs = best_model.predict_proba(
        X_test_scaled
    )

    test_max_prob = np.max(
        test_probs,
        axis=1
    )

    meta_test[:, idx] = test_max_prob


Training LR


KeyboardInterrupt: 

In [ ]:
meta_model = LogisticRegression(
    max_iter=1000
)

meta_model.fit(
    meta_train,
    y_train
)

joblib.dump(
    meta_model,
    "stack_meta_model.joblib"
)

In [ ]:
y_pred = meta_model.predict(
    meta_test
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Stacking Ensemble")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()